### Testing GraphBuilder

#### Node classification - One Graph - Cora

In [7]:
from torch_geometric import datasets
from torch_geometric.loader import DataLoader
from rewiring.model.GCN import GCN
from rewiring.TrainValTestSplitter import TrainValTestSplitter
import torch

dset = datasets.Planetoid(root=f'./../res/datasets/cora', name='cora')
TrainValTestSplitter._cora_split(dset=dset, val_size=0.2, test_size=1)

batch = dset[0]
print(batch.__class__)

dloader = DataLoader(dataset=batch, batch_size=1)
print(dloader.__class__)

splitter = TrainValTestSplitter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN(num_classes=dset.num_classes, 
            num_features=dset.num_features, 
            hidden_size=16)

model.set_classification_task(task='node')
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.NLLLoss()

model.to(device=device)
model.train()
epoch_count = 200
for epoch in range(epoch_count):
    model.train()
    # Clear the gradients
    optimizer.zero_grad()
    # Forward Pass
    out = model(batch.x.to(device), batch.edge_index.to(device))
    # Find the Loss
    train_loss = criterion(out[train_mask], batch.y[train_mask])    
    # Calculate gradients
    train_loss.backward()
    # Update Weights
    optimizer.step()
    print(f"""Epoch: [{epoch}/{epoch_count}]
    Training loss: {train_loss.item():.4f}""")

    ### VALIDATION
    model.eval()
    with torch.no_grad():
        out = model(batch.x.to(device), batch.edge_index.to(device))
        val_loss = criterion(out[val_mask], batch.y[val_mask])
        _, pred = torch.max(out.data, 1)
        correct = (pred[val_mask] == batch.y[val_mask]).float().sum().item()
        acc = correct / val_mask.shape[0]
    
    if (epoch + 1) % 10 == 0:
        print(f"""Epoch: [{epoch+1}/{epoch_count}]
          Training loss: {train_loss.item():.4f},\t Validation loss: {val_loss.item():.4f}
          Accuracy: {acc}""")

<class 'torch_geometric.data.data.Data'>
<class 'torch_geometric.loader.dataloader.DataLoader'>
Epoch: [0/200]
    Training loss: 1.9939
Epoch: [1/200]
    Training loss: 1.9743
Epoch: [2/200]
    Training loss: 1.9525
Epoch: [3/200]
    Training loss: 1.9253
Epoch: [4/200]
    Training loss: 1.8932
Epoch: [5/200]
    Training loss: 1.8570
Epoch: [6/200]
    Training loss: 1.8174
Epoch: [7/200]
    Training loss: 1.7754
Epoch: [8/200]
    Training loss: 1.7314
Epoch: [9/200]
    Training loss: 1.6858
Epoch: [10/200]
          Training loss: 1.6858,	 Validation loss: 1.6395
          Accuracy: 0.6309963099630996
Epoch: [10/200]
    Training loss: 1.6386
Epoch: [11/200]
    Training loss: 1.5904
Epoch: [12/200]
    Training loss: 1.5413
Epoch: [13/200]
    Training loss: 1.4921
Epoch: [14/200]
    Training loss: 1.4432
Epoch: [15/200]
    Training loss: 1.3945
Epoch: [16/200]
    Training loss: 1.3457
Epoch: [17/200]
    Training loss: 1.2967
Epoch: [18/200]
    Training loss: 1.2473
Epo

#### Node Classification - Multiple Graphs - LRGB PascalVOC-SP

In [4]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch_geometric.nn.conv import GCNConv
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import LRGBDataset

from rewiring.model.GCN import GCN
from rewiring.transform.ReduceDiameter import ReduceDiameter

# Define the GCN model
class GCNModel(nn.Module):
    def __init__(self, num_features, num_classes, hidden_size = 16):
        super(GCNModel, self).__init__()
        self.conv1 = GCNConv(num_features, hidden_size)
        self.conv2 = GCNConv(hidden_size, num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = torch.relu(x)
        x = self.conv2(x, edge_index)
        return torch.log_softmax(x, dim=1)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

batch_size = 64
# transform = ReduceDiameter(max_diam=200)
transform = None

# Load the dataset
print("Loading and transforming training dataset...")
train_dataset = LRGBDataset(root=f'./../res/datasets/pascal', name='PascalVOC-SP', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
print("Load and transformation done!")

val_dataset = LRGBDataset(root=f'./../res/datasets/pascal', name='PascalVOC-SP', split='val')
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

test_dataset = LRGBDataset(root=f'./../res/datasets/pascal', name='PascalVOC-SP', split='test')
test_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Create the model
num_features = train_dataset.num_features
num_classes = train_dataset.num_classes
hidden_size = 64
model = GCNModel(num_classes=num_classes,
            num_features=num_features,
            hidden_size=hidden_size).to(device)
# model.set_classification_task('graph')

# Set the loss function and optimizer
criterion = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 3
model.to(device=device)
for epoch in range(num_epochs):

    epoch_train_loss = []
    epoch_val_loss = []

    # Training loop
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        out = model(batch.x, batch.edge_index)
        loss = criterion(out, batch.y)
        loss.backward()
        epoch_train_loss.append(loss.item())

        optimizer.step()

    # Validation loop
    model.eval()
    correct = 0
    sum = len(val_loader.dataset.y)
    for batch in val_loader:
        batch = batch.to(device)
        with torch.no_grad():
            out = model(batch.x, batch.edge_index)
            val_loss = criterion(out, batch.y)
            epoch_val_loss.append(val_loss.item())
            _, pred = torch.max(out.data, 1)
            correct += (pred == batch.y).float().sum().item()

    train_loss = np.array(epoch_train_loss).mean().item()
    val_loss = np.array(epoch_val_loss).mean().item()
    acc = correct / sum

    print(f"""Epoch: [{epoch+1}/{num_epochs}] \
        Training loss: {train_loss:.4f} \
        Validation loss: {val_loss:.4f} \
        Accuracy: {acc:.4f}""")

print("Training finished!")
print("Starting testing...")
model.eval()
correct = 0
epoch_test_loss = []
sum = len(test_loader.dataset.y)
print(f'Test element count: {sum}')
for batch in test_loader:
    batch = batch.to(device)
    with torch.no_grad():
        out = model(batch.x, batch.edge_index)
        test_loss = criterion(out, batch.y)
        epoch_test_loss.append(test_loss.item())
        _, pred = torch.max(out.data, 1)
        correct += (pred == batch.y).float().sum().item()

acc = correct / sum
test_loss = np.array(epoch_test_loss).mean().item()

print(f"""Loss: {test_loss:.4f} \
        Accuracy: {acc:.4f}""")

Loading and transforming training dataset...
Load and transformation done!
Epoch: [1/3]         Training loss: 10.8140         Validation loss: 2.1176         Accuracy: 0.6930
Epoch: [2/3]         Training loss: 1.7925         Validation loss: 1.6418         Accuracy: 0.6964
Epoch: [3/3]         Training loss: 1.5851         Validation loss: 1.5547         Accuracy: 0.6908
Training finished!
Starting testing...
Test element count: 685013
Loss: 1.5547         Accuracy: 0.6908


#### Graph Classification - Proteins, Mutag

In [59]:
from torch_geometric import datasets
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import TwoHop
from rewiring.model.GCN import GCN
from rewiring.TrainValTestSplitter import TrainValTestSplitter
from rewiring.transform.ReduceDiameter import ReduceDiameter
from rewiring.transform.AddRandomEdges import AddRandomEdges

import torch
import numpy as np
import logging

logging.basicConfig(level=logging.INFO)

transform = None
# transform = AddRandomEdges(min_density=0.4)
# transform = ReduceDiameter(max_diam=26)
# transform = TwoHop()

# data = datasets.TUDataset(root=f'./../res/datasets/proteins', name='PROTEINS', transform=transform, use_node_attr=True, use_edge_attr=True)
# data = datasets.TUDataset(root=f'./../res/datasets/mutag', name='MUTAG', transform=transform, use_node_attr=True, use_edge_attr=True)
# data = datasets.TUDataset(root=f'./../res/datasets/collab', name='COLLAB', transform=transform, use_node_attr=True, use_edge_attr=True)
print(f"Number of graphs: {len(batch)}")

val_ratio = 0.2
train_dataset, val_dataset, test_dataset = TrainValTestSplitter.graph_class_split(batch, val_size=val_ratio, test_size=1)
val_count = len(val_dataset)

batch_size = 64

train_dloader: DataLoader = DataLoader(train_dataset, 
                                       shuffle=True, 
                                       batch_size=batch_size)

val_dloader: DataLoader = DataLoader(val_dataset, 
                                      shuffle=False, 
                                      batch_size=batch_size)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Creating model...")
model = GCN(num_classes=batch.num_classes, 
            num_features=batch.num_features, 
            hidden_size=16)
print("Model created!")
model.set_classification_task(task='graph')
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.NLLLoss()

model.to(device=device)

acc_list = []
train_losses = []
val_losses = []

print("Starting training...")

epoch_count = 50
for epoch in range(1, epoch_count + 1):

    epoch_train_loss = []
    epoch_val_loss = []

    ### TRAINING
    model.train()
    for batch in train_dloader:
        # Clear the gradients
        optimizer.zero_grad()
        # Forward Pass
        out = model(batch.x.to(device), batch.edge_index.to(device), batch.batch.to(device))
        # Find the Loss
        train_loss = criterion(out, batch.y)
        epoch_train_loss.append(train_loss.item())    
        # Calculate gradients
        train_loss.backward()
        # Update Weights
        optimizer.step()

    ### VALIDATION
    model.eval()
    correct = 0
    for batch in val_dloader:
        with torch.no_grad():
            out = model(batch.x.to(device), batch.edge_index.to(device), batch.batch.to(device))
            val_loss = criterion(out, batch.y)
            epoch_val_loss.append(val_loss.item())
            _, pred = torch.max(out.data, 1)
            correct += (pred == batch.y).float().sum().item()

    acc = correct / val_count
    acc_list.append(acc)

    epoch_train_loss = np.array(epoch_train_loss).mean().item()
    epoch_val_loss = np.array(epoch_val_loss).mean().item()
    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
        
    print(f"""Epoch: [{epoch}/{epoch_count}] \
        Training loss: {epoch_train_loss:.4f} \
        Validation loss: {epoch_val_loss:.4f} \
        Accuracy: {acc:.4f}""")

Number of graphs: 5000
Creating model...
Model created!
Starting training...


AttributeError: 'NoneType' object has no attribute 'to'

In [1]:
from torch_geometric import datasets
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import TwoHop
from rewiring.model.GCN import GCN
from rewiring.TrainValTestSplitter import TrainValTestSplitter
from rewiring.transform.ReduceDiameter import ReduceDiameter
from rewiring.transform.AddRandomEdges import AddRandomEdges
from rewiring.pipeline.GraphClassificationPipeline import GraphClassificationPipeline

import torch
import numpy as np
import logging

logging.basicConfig(level=logging.INFO)

transform = None
# transform = AddRandomEdges(min_density=0.4)
# transform = ReduceDiameter(max_diam=26)
# transform = TwoHop()

batch = datasets.TUDataset(root=f'./../res/datasets/proteins', name='PROTEINS', transform=transform, use_node_attr=True, use_edge_attr=True)
# data = datasets.TUDataset(root=f'./../res/datasets/mutag', name='MUTAG', transform=transform, use_node_attr=True, use_edge_attr=True)
# data = datasets.TUDataset(root=f'./../res/datasets/collab', name='COLLAB', transform=transform, use_node_attr=True, use_edge_attr=True)
print(f"Number of graphs: {len(batch)}")

val_ratio = 0.2
train_dataset, val_dataset, test_dataset = TrainValTestSplitter.graph_class_split(batch, val_size=val_ratio, test_size=1)
val_count = len(val_dataset)

batch_size = 64

train_dloader: DataLoader = DataLoader(train_dataset, 
                                       shuffle=True, 
                                       batch_size=batch_size)

val_dloader: DataLoader = DataLoader(val_dataset, 
                                      shuffle=False, 
                                      batch_size=batch_size)

print("Creating model...")
model = GCN(num_classes=batch.num_classes, 
            num_features=batch.num_features, 
            hidden_size=16)
model.set_classification_task(task='graph')
print("Model created!")

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.NLLLoss()

pipeline = GraphClassificationPipeline(
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    model=model,
    criterion=criterion,
    optimizer=optimizer
)

pipeline.train_validate(
    train_dloader=train_dloader,
    val_dloader=val_dloader,
    epochs=50,
    log=True
)

Number of graphs: 1113
Creating model...
Model created!
Epoch: [0/50]         Training loss: 0.7069         Validation loss: 0.7049         Accuracy: 0.5740
Epoch: [1/50]         Training loss: 0.6760         Validation loss: 0.6850         Accuracy: 0.5695
Epoch: [2/50]         Training loss: 0.6623         Validation loss: 0.6841         Accuracy: 0.5695
Epoch: [3/50]         Training loss: 0.6582         Validation loss: 0.6897         Accuracy: 0.5785
Epoch: [4/50]         Training loss: 0.6698         Validation loss: 0.7252         Accuracy: 0.5695
Epoch: [5/50]         Training loss: 0.6728         Validation loss: 0.6692         Accuracy: 0.5919
Epoch: [6/50]         Training loss: 0.6601         Validation loss: 0.7026         Accuracy: 0.5605
Epoch: [7/50]         Training loss: 0.6670         Validation loss: 0.6630         Accuracy: 0.5785
Epoch: [8/50]         Training loss: 0.6550         Validation loss: 0.6618         Accuracy: 0.5785
Epoch: [9/50]         Training loss

([0.7069067656993866,
  0.676014108317239,
  0.6622718700340816,
  0.6581740230321884,
  0.6697554801191602,
  0.6727905528885978,
  0.6600886540753501,
  0.666987201997212,
  0.6549664310046605,
  0.6479509132249015,
  0.6454114658491952,
  0.6385838857718876,
  0.6342847389834267,
  0.6293637369360242,
  0.6356937118938991,
  0.6230646201542446,
  0.6139719997133527,
  0.6170967561858041,
  0.6155451578753335,
  0.6069285529000419,
  0.5998474189213344,
  0.601343993629728,
  0.5997765575136457,
  0.6029577553272247,
  0.5981541914599282,
  0.6025115890162331,
  0.6044416299888066,
  0.5884752827031272,
  0.5836358283247266,
  0.5837774191583905,
  0.579441059912954,
  0.5754318535327911,
  0.579894802400044,
  0.5729251482657024,
  0.5855588018894196,
  0.5732604478086744,
  0.5719455480575562,
  0.5805698015860149,
  0.5639822781085968,
  0.562117161495345,
  0.5585454191480365,
  0.5618557780981064,
  0.5621865711041859,
  0.5653896714959826,
  0.5568545801298959,
  0.548797036920

In [4]:
import logging
from torch_geometric import datasets
from rewiring.TrainValTestSplitter import TrainValTestSplitter

logging.basicConfig(level=logging.DEBUG)

dset = datasets.Planetoid(root=f'./../res/datasets/citeseer', name='CiteSeer')
TrainValTestSplitter.single_graph_split(dset=dset, val_size=0.2, test_size=1)

DEBUG:rewiring.TrainValTestSplitter:Proceeding with single graph splitting method
DEBUG:rewiring.TrainValTestSplitter:Train mask: [1996 1106 2226 ... 1382 2560 2106]
DEBUG:rewiring.TrainValTestSplitter:Val mask: [ 455 2374 1351 2451   65 1671  994 2379 2250 2373  611 1712  487 2644
  552 2356  544  745  777  730  366  828 1399 2078   30 1758 2491  710
  933  595 2163 1622 2420   44  604 1083 1346 2544 2688 1403 2589  700
 1729  705 2505  497 2279 1173  952  536   48 1380 2208 2088 2378 1042
  967 1015  254  278  980 1744  478   18 2357   36 1388 2647 1683  433
 2484 2696 2440  510  588 1889 1714 2110 1348 2443 2013 1748 1020  539
  911 2367  575  185  861 1036 1105 1400  126 1772  881  919 1215  436
 1361   84 1121 1353 1251 2025 2603 2185  819 1952  114 2316  666 1392
  754 1157 2114 2477 1620 2471 1526  605  746 1551  297  230 1433 2239
 1689  560  723 1141 1111  750 1920  196 1274 1116 1867 1369   81 2529
 2430 2271 2171 1589 1727 1700 2650  305 1876 1741 1056 1270  139 1656
 1246  

(array([1996, 1106, 2226, ..., 1382, 2560, 2106]),
 array([ 455, 2374, 1351, 2451,   65, 1671,  994, 2379, 2250, 2373,  611,
        1712,  487, 2644,  552, 2356,  544,  745,  777,  730,  366,  828,
        1399, 2078,   30, 1758, 2491,  710,  933,  595, 2163, 1622, 2420,
          44,  604, 1083, 1346, 2544, 2688, 1403, 2589,  700, 1729,  705,
        2505,  497, 2279, 1173,  952,  536,   48, 1380, 2208, 2088, 2378,
        1042,  967, 1015,  254,  278,  980, 1744,  478,   18, 2357,   36,
        1388, 2647, 1683,  433, 2484, 2696, 2440,  510,  588, 1889, 1714,
        2110, 1348, 2443, 2013, 1748, 1020,  539,  911, 2367,  575,  185,
         861, 1036, 1105, 1400,  126, 1772,  881,  919, 1215,  436, 1361,
          84, 1121, 1353, 1251, 2025, 2603, 2185,  819, 1952,  114, 2316,
         666, 1392,  754, 1157, 2114, 2477, 1620, 2471, 1526,  605,  746,
        1551,  297,  230, 1433, 2239, 1689,  560,  723, 1141, 1111,  750,
        1920,  196, 1274, 1116, 1867, 1369,   81, 2529, 2430,